[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/snowflake/snowflake.ipynb)

# Snowflake Growth

This example computes a snowflake growing: a seed of ice in undercooled water growing into a six-armed dendrite. Water cooled below its melting point freezes onto any ice it touches, and freezing releases latent heat, which warms the water at the ice's edge and slows the freezing there. A bump on the edge reaches into colder water and sheds its heat to more sides than the flat edge around it, so it grows faster: the tips outrun the sides, and each tip sprouts branches of its own. The ice's surface energy depends on how its edge lies against the crystal's axes, six-fold for ice, and steers the fastest tips along six directions, the six arms of a snowflake. What drives the freezing is the entropy it produces: the latent heat a unit of ice gives up raises the entropy of the colder water around it by more than the ice's own entropy falls, so freezing pays wherever the water is colder than its melting point, and the push it makes comes from the same latent heat that warms the water and holds the freezing back. The ice is a phase field, an order parameter at each place, one in ice and nothing in water, with a smooth edge between them, far wider than real ice's so that the mesh resolves it.

The algebra is the plane joined with three directions that have no metric, for the volume, the ice and the heat a parcel of water holds, `Algebra("x+y+") * Algebra("v0i0h0")`. A parcel is a point weighed by its volume and offset along the other two by the ice and the heat it holds, which `mv.i &` and `mv.h &` read off it. Freezing is one fixed direction among these amounts, `freezing = ice + latent_heat * heat`, ice gained and its latent heat released, and two planes read what it does. The plane that reads a parcel's energy, `mv.h - latent_heat * mv.i`, pairs with it to nothing: freezing only moves energy from the latent store into the heat. The entropy's slope over the amounts, `mv.h / absolute_temperatures - latent_heat / absolute_melting * mv.i`, pairs with it to the entropy that freezing a unit of ice produces, positive in undercooled water and nothing at the melting temperature: energy kept and entropy produced are two pairings with one direction. Times the melting temperature, that pairing pushes the phase towards ice through the slope of the ice a parcel holds, a smooth step in the phase that is flat at water, so that the water stays metastable at any undercooling.

A gradient wedged with the amount it acts on is a drive, which a material map takes to the flux it drives; the heat's drive is conducted, and the phase's drive paired with its own flux, `drive & spreading(drive)`, is the squared steepness of the ice's edge, with no metric square taken. The edge's width varies with its normal through the turn from the crystal's axis to the steepness, `turn = steepness * crystal_axis`, a scaled rotor whose sixth power has the six-fold `cos(6 angle)` as its scalar part, with no angle taken. The derivative of such an anisotropic gradient energy is notoriously fiddly by hand; here it is written out, the six-fold term's slope being the rotor's fifth power, `6 * crystal_axis * turn ** 5`, and each vertex's share its hat's drive paired with the flux that slope drives. JAX only compiles the stepping.

In the vector-calculus notation of phase-field models, with $\phi$ the phase, $T$ the absolute temperature, $\theta$ the angle of the edge's normal $\nabla\phi$ from the crystal's axis, and the heat capacity one, the energy is $e = T - L\,p(\phi)$ with the step $p(\phi) = \phi^3(10 - 15\phi + 6\phi^2)$, and the entropy is $S = \int s - \tfrac{1}{T_m}\big(\tfrac12\,\epsilon(\theta)^2\,|\nabla\phi|^2 + W\phi^2(1-\phi)^2\big)\,dA$, with $s = \ln T - \tfrac{L}{T_m}\,p(\phi)$ and the width $\epsilon(\theta) = \bar\epsilon\,(1 + \delta\cos 6\theta)$. The phase relaxes up the entropy at fixed energy, $\tau\,\partial_t\phi = T_m\,\delta S/\delta\phi|_e = \nabla\cdot(\epsilon^2\nabla\phi) - \partial_x(\epsilon\,\epsilon'\,\partial_y\phi) + \partial_y(\epsilon\,\epsilon'\,\partial_x\phi) - 2W\phi(1-\phi)(1-2\phi) + L\,\tfrac{T_m - T}{T}\,p'(\phi)$, and the heat follows as $\partial_t T = \nabla^2 T + L\,\partial_t p(\phi)$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples,jax] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import jax
import numpy as np

from numga import Algebra, stack
from numga.backend.jax import JaxContext
from numga.sparse import spdiag
from examples.util import shapes
from examples.util.animation import show_states
from examples.thermodynamics.snowflake import render, scenarios

# The plane, and the volume, the ice and the heat a parcel of water holds: each a direction of its own, degenerate,
# since nothing measures a volume, an amount of ice or of heat against a length.
ga = Algebra("x+y+") * Algebra("v0i0h0")
mv = JaxContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Point = ga.gatype.antivector()                                                # Antivector-
# An amount a cell holds, intrinsic like the join that measures its area: the area, or the energy in it.
Amount = ga.gatype.antipseudoscalar()                                         # Antipseudoscalar-
# The planes reading a place's run across and up; the drives, a gradient wedged with the ice or the heat.
Upright = ga.gatype.from_blades("x y")                                        # Vector+
Drive = ga.gatype.from_blades("xi yi xh yh")                                  # Bivector+
# The plane's two directions, across and up.
directions = stack((mv.x.dual(), mv.y.dual()))                                # [2] Point-
# A unit of volume, the weight of a place, a unit of ice and a unit of heat.
volume = mv.v.dual()                                                          # [] Point-
ice = mv.i.dual()                                                             # [] Point-
heat = mv.h.dual()                                                            # [] Point-

## The dish of water

The water fills a round dish, meshed in nearly equilateral triangles laid in staggered rings around its middle. The mesh has no six-fold symmetry of its own; a lattice of equilateral triangles would have one, and would carry its six directions into the crystal's growth. The water starts cold, at temperature zero, with a seed of ice in its middle. Temperatures are measured from the cold water's in units of its undercooling, so that ice melts at one, and times are in units in which heat diffuses across a unit of length in a unit of time. The water's state is two fields: its parcels, and the phase, an order parameter rather than an amount, one in ice and nothing in water. The ice a parcel holds follows the phase as a smooth step, `held_ice(phase)`, flat at water and at ice. A field is interpolated across each triangle by its corners' hats, whose gradients are planes built from the triangle's sides by joins alone. The triangle's area stays intrinsic, like the join of its sides that measures it, and so does what a cell holds: an energy's change at a vertex is an amount, and only over the cell's area does it become a rate.

In [ ]:
def held_ice(phase: Scalar) -> Scalar:
    """The ice a parcel holds at this phase `Scalar+[vertices]`: none in water and all in ice, a smooth step between
    whose slope, `30 * phase ** 2 * (1 - phase) ** 2`, vanishes at both."""
    return phase ** 3 * (10 - 15 * phase + 6 * phase ** 2)


# The dish, its vertices places of unit volume.
topology, coordinates = shapes.concentric_disk(np.array([scenarios.dish_radius]), scenarios.n_divisions)
vertices = (volume + mv(Upright, coordinates).dual()).field()                 # Point-[vertices]
# The seed: the phase one in ice and nothing in water, with a smooth edge between.
phase = mv.scalar(scenarios.seed_phase(coordinates)[:, None]).field()        # Scalar+[vertices]
# The water at the start: unit volume, the ice its phase holds, and no heat yet, its type holding the heat's direction
# for the run to fill.
parcels = volume + ice * held_ice(phase) + 0 * heat                           # Point-[vertices]

sides = topology.boundary[2, 1] * (topology.d[0] * vertices)                  # [corners] Point-[faces]
# Twice a triangle's area: two of its sides joined, at unit volume, ice and heat.
double_areas = (((sides[0] & sides[1]) & volume) & ice) & heat                # Scalar-[faces]
areas = double_areas / 2                                                      # Scalar-[faces]
# A corner's hat gradient: the plane through the facing side, along volume, ice and heat, over twice the area, which
# reads one from the side to the corner.
gradients = (volume & ice & heat & sides) / double_areas                      # [corners] Plane+[faces]
# Each vertex's share of the area around it.
corners = topology.incidence[2, 0]                                            # [corners] Scalar+[faces <- vertices]
cells = (~corners * (areas / 3)).sum(axis=0)                                  # Scalar-[vertices]
render.draw_seed(topology, vertices, parcels);

The seed close up in the dish's triangles: a disc of ice, pale, a fifth of a unit across, in the cold water, dark. The innermost ring of the mesh is a hexagon, and each ring outward holds a few more vertices than the one inside it, so the rings soon fall out of step with any six-fold pattern.

## Freezing and its entropy

Freezing moves a parcel along one direction among its amounts, `freezing = ice + latent_heat * heat`: ice gained, and its latent heat released into the heat. Ice is taken at water's density, so freezing leaves the volume alone, `mv.v & freezing == 0`, and ice and water share one heat capacity and one conductivity. The plane that reads a parcel's energy, its heat less the latent heat its ice has given up, `energy = mv.h - latent_heat * mv.i`, pairs with freezing to nothing: freezing only moves energy from the latent store into the heat. The entropy's slope over a parcel's amounts is a plane too, `mv.h / absolute_temperatures - latent_heat / absolute_melting * mv.i`: heat raises the entropy by one over the absolute temperature, and ice holds the latent heat over the melting temperature less entropy than water. Paired with freezing it gives the entropy that freezing a unit of ice produces, `latent_heat * (1 / absolute_temperatures - 1 / absolute_melting)`: positive in undercooled water, nothing at the melting temperature, and made of the same latent heat that warms the water. Only the one over the temperature needs an absolute scale. Water's latent heat would warm it by about 80 kelvin, so a latent heat of 1.8 undercoolings makes the undercooling about 44 kelvin, and puts water's melting point, at 273 kelvin, six undercoolings above absolute zero.

In [ ]:
# The latent heat's warming per unit of ice, the melting temperature, with the cold water at nothing, and absolute
# zero on that scale; the melting temperature above absolute zero.
latent_heat, melting_temperature, absolute_zero = 1.8, 1.0, -5.0
absolute_melting = melting_temperature - absolute_zero
# Freezing: ice gained, and its latent heat released.
freezing = ice + latent_heat * heat                                           # [] Point-
# The plane reading the energy a parcel holds: its heat, less the latent heat its ice has given up.
energy = mv.h - latent_heat * mv.i                                            # [] Plane+

## The edge's free energy

The phase also carries the free energy of the ice's edge, summed over the dish: a double well with its minima at water and ice, and a gradient energy, `areas * widths ** 2 * (drive & spreading(drive)) / 2` over the triangles, which makes the edge between them cost energy and sets its width. Over the melting temperature, that free energy is what the edge takes off the entropy. The gradient energy pairs the phase's drive, minus its steepness wedged with the ice, with the flux the drive pushes, and so measures the squared steepness without a metric square; the same material map conducts the heat. The edge's width varies six-fold with its normal, through the sixth power of the turn from the crystal's axis to the steepness, so that the ice's surface costs a little more where its normal lies along one of the crystal's six axes.

In [ ]:
# The phase's relaxation time, its edge's mean width, the six-fold anisotropy of that width, the steepness below which
# the anisotropy fades, where the phase is flat, and the double well's height.
relaxation_time, mean_width, anisotropy, regularization, well_height = 3e-4, 0.0082, 0.025, 1.0, 1 / 6
# The crystal's axis, a plane whose normal, up, one arm grows along.
crystal_axis = mv.y                                                           # [] Plane+

In [ ]:
# A field's gradient across each triangle: its corners' values on their hats' gradients.
gradient = (spdiag(gradients) * corners).sum(axis=0) * Scalar                 # (Plane+ <- Scalar+)[faces <- vertices]
# The material map, the same for the heat and the ice at unit rate: each of the plane's directions swept through the
# amount, read by a drive and driving the amount along it.
sweeps = stack((directions & heat, directions & ice))                         # [amounts, 2] Flux-
spreading = (sweeps * (sweeps & Drive)).sum(axis=-1).sum(axis=0)             # [] Flux- <- Drive+

The edge's free energy's slope with respect to the phase at each vertex holds the phase back, and it is written out. The six-fold term's slope with respect to the steepness is the turn's fifth power, `6 * crystal_axis * turn ** 5`: the derivative of the six-fold factor extended off the unit normals as a homogeneous polynomial, as the Cahn–Hoffman vector of materials science is the derivative of a surface energy extended so. Each vertex's share of the gradient energy's change is its hat's drive paired with the energy's slope, a flux: twice the phase's own flux, `spreading(drive)`, is the slope of its squared steepness.

The six-fold term is the one place where the plane's own metric enters. The squared steepness pairs a drive with its flux and needs none, but the angle between the edge's normal and the crystal's axis does: the turn `steepness * crystal_axis` is a geometric product in `x+y+`, as it should be, since the crystal's six directions are a fact about angles in its lattice. Its slope therefore comes out as a plane, measured by that metric, where the gradient energy's other slope is a flux, and the material map turns it into one, `spreading(-(slope ^ mv.i))`. That is right because the spreading here is the plane's metric itself, the same in every direction; with a gradient energy that measured steepness unevenly, the turn would have to be measured in that energy's metric too.

In [ ]:
# Each corner's hat's drive on the ice.
ice_drives = -(gradients ^ mv.i)                                              # [corners] Drive+[faces]


def edge_slope(phase: Scalar, anisotropy: float) -> Amount:
    """The derivative of the edge's free energy with respect to the phase at each vertex `Scalar-[vertices]`, at this
    phase `Scalar+[vertices]`, with this anisotropy."""
    steepness = gradient(phase)                                               # Plane+[faces]
    # The phase's drive and the flux it drives; paired, the edge's squared steepness, whose slope is twice the flux.
    drive = -(steepness ^ mv.i)                                               # Drive+[faces]
    flux = spreading(drive)                                                   # Flux-[faces]
    squared_steepness = drive & flux                                          # Scalar+[faces]
    turn = steepness * crystal_axis                                           # Even+[faces]
    third_power = turn ** 3                                                   # Even+[faces]
    six_fold = third_power.scalar_product(third_power)                        # Scalar+[faces]
    reach = squared_steepness + regularization ** 2                           # Scalar+[faces]
    widths = mean_width * (1 + anisotropy * six_fold / reach ** 3)            # Scalar+[faces]
    # The six-fold term's slope with respect to the steepness, the turn's fifth power: a plane, measured in the plane's
    # metric, and the flux the material map, that same metric here, makes of it.
    slope = 6 * crystal_axis * turn ** 5                                      # Plane+[faces]
    slope_flux = spreading(-(slope ^ mv.i))                                   # Flux-[faces]
    # The widths' change as the edge turns; the gradient energy's slope, the widths squared times the flux and that
    # change times the steepness squared.
    turning = mean_width * anisotropy * (slope_flux / reach ** 3 - 6 * six_fold * flux / reach ** 4)   # Flux-[faces]
    pull = widths ** 2 * flux + squared_steepness * widths * turning          # Flux-[faces]
    # Each vertex's share of the gradient energy's change: its hat's drive paired with the energy's slope.
    shares = (~corners * (areas * (ice_drives & pull))).sum(axis=0)           # Scalar-[vertices]
    well_slope = 2 * well_height * phase * (1 - phase) * (1 - 2 * phase)      # Scalar+[vertices]
    return shares + cells * well_slope

## Growing the snowflake

The phase relaxes up the total entropy at fixed energy, over the relaxation time. Where the ice a parcel holds follows the phase, the push towards ice is the entropy that freezing produces, times the slope of the ice held; times the melting temperature it is a free energy, for a unit of ice the latent heat times the undercooling over the absolute temperature, which the edge's slope holds back. The smooth step the ice held follows is a device of the phase field, not a fact about ice: its slope vanishes at water, and the push with it, so the water stays a local minimum of the free energy at any undercooling, if a shallow one where it is coldest, metastable until the ice's edge reaches it; with the ice held simply equal to the phase, the push would freeze the cold water wherever it stands. Each step, every parcel takes up the ice its phase now holds, so the heat it gains is the latent heat of that ice, and the heat is conducted: each hat's drive on the heat, through the same material map, is read by each other hat for the heat it carries out of that hat's cell. The rim lets no heat through, and the run lasts until the snowflake's tips reach about nine tenths of the way to it. JAX compiles the step and scans the steps between frames. The cells below take about fifteen seconds, most of it stepping the water.

In [ ]:
# The time step and the steps between frames; the frames the snowflake takes until its tips reach about nine tenths
# of the dish's radius; the frames' duration in milliseconds.
dt, stride = 2e-4, 20
n_snowflake_frames = 75
frame_duration_ms = 40

# The heat each vertex's cell loses to each vertex's warmth: each hat's drive on the heat conducted, and read by the
# other hats for its heat, over the triangles.
heat_drives = -(gradients ^ mv.h)                                             # [corners] Drive+[faces]
local = areas * (mv.h & (gradients[:, None] ^ spreading(heat_drives[None, :])))   # [corners, corners] Scalar-[faces]
conduction = (~corners[:, None] * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)   # Scalar-[vertices <- vertices]

In [ ]:
def growth(parcels: Point, phase: Scalar, anisotropy: float, n_frames: int) -> Iterator[tuple[Point, Scalar]]:
    """The water's parcels `Point-[vertices]` and the phase `Scalar+[vertices]` at each of `n_frames` frames, from
    these, the ice's edge's width varied six-fold by this anisotropy."""

    def step(state: tuple[Point, Scalar], _) -> tuple[tuple[Point, Scalar], None]:
        parcels, phase = state
        temperatures = mv.h & parcels                                         # Scalar+[vertices]
        # The entropy's slope over a parcel's amounts: heat raises it by one over the absolute temperature, and ice
        # holds the latent heat over the absolute melting temperature less of it than water.
        absolute_temperatures = temperatures - absolute_zero                  # Scalar+[vertices]
        entropy_slope = mv.h / absolute_temperatures - latent_heat / absolute_melting * mv.i   # Plane+[vertices]
        # Paired with freezing, the entropy freezing a unit of ice produces; times the melting temperature, a free
        # energy, and times the slope of the ice held, the push to freeze.
        held_slope = 30 * phase ** 2 * (1 - phase) ** 2                       # Scalar+[vertices]
        push = absolute_melting * cells * (entropy_slope & freezing) * held_slope   # Scalar-[vertices]
        # The phase relaxes up the total entropy: the push, less the edge's free energy's slope.
        next_phase = phase + dt * (push - edge_slope(phase, anisotropy)) / (relaxation_time * cells)   # Scalar+[vertices]
        # Each parcel takes up the ice its phase now holds, its latent heat with it, and the heat is conducted.
        ice_gained = held_ice(next_phase) - (mv.i & parcels)                  # Scalar+[vertices]
        conducted = -(conduction * temperatures) / cells                      # Scalar+[vertices]
        return (parcels + ice_gained * freezing + dt * conducted * heat, next_phase), None

    # The steps between two frames, compiled and scanned.
    @jax.jit
    def strided(state: tuple[Point, Scalar]) -> tuple[Point, Scalar]:
        return jax.lax.scan(step, state, None, length=stride)[0]

    state = (parcels, phase)
    for _ in range(n_frames):
        yield state
        state = strided(state)


snowflake = show_states(growth(parcels, phase, anisotropy, n_snowflake_frames), render.animate, topology, vertices,
                        name="snowflake", duration_ms=frame_duration_ms)

The seed grows first into a hexagonal plate, its corners on the crystal's six axes and its faces turned between them, where the edge costs least. The corners reach into colder water and grow into arms, and each arm's sides break into branches, which grow along the crystal's axes too. The pale halo around the ice is the latent heat its freezing released, spreading into the cold water, dark. The arms grow at much the same pace, one a little ahead, and the run ends as the farthest tip nears the dish's rim, the outer tenth of the dish still water; the finished crystal is held a moment before the loop starts over.

## Entropy over the growth

The total entropy is the parcels' entropy, summed over the cells, less the edge's free energy over the melting temperature. The stepping holds the energy by construction, moving parcels only along freezing and passing heat between cells, but nothing in it holds the entropy to anything: that the entropy rises is the second law, and it comes from the phase moving up the entropy's slope and the heat flowing from warm water to cold.

In [ ]:
def edge_free_energy(phase: Scalar, anisotropy: float) -> Amount:
    """The free energy of the ice's edge `[] Scalar-` at this phase `Scalar+[vertices]`, with this anisotropy: the
    gradient energy over the triangles and the double well over the cells."""
    steepness = gradient(phase)                                               # Plane+[faces]
    drive = -(steepness ^ mv.i)                                               # Drive+[faces]
    squared_steepness = drive & spreading(drive)                              # Scalar+[faces]
    third_power = (steepness * crystal_axis) ** 3                             # Even+[faces]
    six_fold = third_power.scalar_product(third_power)                        # Scalar+[faces]
    widths = mean_width * (1 + anisotropy * six_fold / (squared_steepness + regularization ** 2) ** 3)   # Scalar+[faces]
    well = well_height * phase ** 2 * (1 - phase) ** 2                        # Scalar+[vertices]
    return (areas * widths ** 2 * squared_steepness / 2).sites.sum() + (cells * well).sites.sum()


def total_entropy(parcels: Point, phase: Scalar) -> Amount:
    """The snowflake's total entropy `[] Scalar-` with these parcels `Point-[vertices]` at this phase
    `Scalar+[vertices]`."""
    # Each parcel's entropy: the logarithm of its absolute temperature, less the latent heat over the absolute melting
    # temperature for each unit of its ice.
    parcel_entropies = ((mv.h & parcels) - absolute_zero).log() - latent_heat / absolute_melting * (mv.i & parcels)
    return (cells * parcel_entropies).sites.sum() - edge_free_energy(phase, anisotropy) / absolute_melting


entropies = stack([total_entropy(*state) for state in snowflake])             # [frames] Scalar-
render.draw_entropy(dt * stride * np.arange(n_snowflake_frames), entropies);

The entropy gained since the seed was set in the water, against time: it rises throughout, and faster as the ice's edge lengthens and releases more latent heat into the cold water.

## Without the crystal's preference

The same seed in the same water grows again with the anisotropy taken away, its edge's width the same whichever way the edge faces, until its tips too reach about nine tenths of the way to the rim. This cell takes about twenty seconds, most of it stepping the water.

In [ ]:
# The frames the bush takes until its tips reach about nine tenths of the dish's radius.
n_isotropic_frames = 111
isotropic = show_states(growth(parcels, phase, 0.0, n_isotropic_frames), render.animate, topology, vertices,
                        name="snowflake_isotropic", duration_ms=frame_duration_ms)

Without the crystal's preference the edge is as unstable as before: its bumps still outrun their sides and split. But no direction is favoured, and the seed grows into a bush of fingers, their number and their directions set by small irregularities, the mesh's among them, with no six-fold order, and it takes half as long again as the snowflake to reach as far. The snowflake's six arms come from the crystal's anisotropy, not from the mesh.

In [ ]:
# checks
# Freezing moves energy from the latent store into the heat: the energy plane pairs with it to nothing.
np.testing.assert_allclose((energy & freezing).to_array(), 0.0, atol=1e-15)
# The rim holds the heat in, so in both runs the water's energy summed over the cells stays what it was: the heat
# gained is the latent heat of the ice gained.
for states in (snowflake, isotropic):
    first, last = ((cells * (energy & parcels)).sites.sum().to_array() for parcels, _ in (states[0], states[-1]))
    np.testing.assert_allclose(last, first, rtol=1e-10)
# The second law: the total entropy rises from every frame to the next.
assert np.all(np.diff(entropies.to_array()) > 0)
# The seed's plate turns its faces between the crystal's axes, while without the anisotropy the seed's edge faces every
# way alike: the edge's mean six-fold cos(6 angle) from the crystal's axis, weighted by its squared steepness, at the
# frame where the plate shows.
plate_frame = 2
alignments = []
for states in (snowflake, isotropic):
    steepness = gradient(states[plate_frame][1])                              # Plane+[faces]
    squared_steepness = -(steepness ^ mv.i) & spreading(-(steepness ^ mv.i))  # Scalar+[faces]
    third_power = (steepness * crystal_axis) ** 3                             # Even+[faces]
    six_fold = third_power.scalar_product(third_power)                        # Scalar+[faces]
    alignments.append(((areas * six_fold / (squared_steepness + regularization ** 2) ** 2).sites.sum()
                       / (areas * squared_steepness).sites.sum()).to_array())
assert alignments[0] < -0.25
assert abs(alignments[1]) < 0.1